# 🤖 Machine Learning Pipeline & MLflow Tracking
## Logistic Regression, Random Forest, and XGBoost with Calibration

In this notebook, we train, evaluate, tune, and calibrate multiple churn classification models:
- **Baseline Models**: Logistic Regression, Random Forest, XGBoost
- **Leakage Prevention**: Stratified splitting before preprocessor fitting
- **Comprehensive Metrics**: ROC-AUC, PR-AUC, F1-Score, Recall, Precision, Log Loss, Brier Score
- **Calibration**: Isotonic & Sigmoid Platt Scaling
- **MLflow Tracking**: Experiment tracking and artifact registry


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from src.models.train import train_and_evaluate_all, load_splits
from src.models.metrics import metrics_row
from src.models.calibration import evaluate_calibrators, build_tuned_estimator

splits, results, pipelines, skipped = train_and_evaluate_all(include_engineered=True)
print(f"Trained {len(results)} models successfully!")


### 1. Model Comparison Table (Validation Set)


In [ ]:
comparison_rows = [metrics_row(name, m) for name, m in results.items()]
comp_df = pd.DataFrame(comparison_rows).sort_values("roc_auc", ascending=False)
comp_df


### 2. Probability Calibration Analysis


In [ ]:
cal_results, best_cal_name = evaluate_calibrators()
print(f"Best Calibration Method: {best_cal_name}")
pd.DataFrame([
    {"Method": name, "Brier Score": res["brier"], "ROC-AUC": res["roc_auc"]}
    for name, res in cal_results.items()
])
